In [ ]:
!pip install --quiet pydantic-ai nest_asyncio

In [ ]:
from importlib.metadata import version

try:
    import pydantic_ai
    import nest_asyncio
    print("✅ pydantic-ai instalado. Versión:", version("pydantic-ai"))
    print("📝 Anota esta versión en tus apuntes.")
except ImportError as error:
    print("❌ Algo falló en la instalación:", error)
    print("Vuelve a ejecutar la celda anterior sin --quiet para ver el error completo.")

In [ ]:
import os
from google.colab import userdata

try:
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    print("✅ API key cargada correctamente.")
except Exception as error:
    print("❌ No se pudo leer el secreto OPENROUTER_API_KEY.")
    print("Revisa que exista con ese nombre exacto y que tenga activado")
    print("el interruptor 'Acceso desde el notebook'.")
    print("Detalle técnico:", error)

In [ ]:
import nest_asyncio

nest_asyncio.apply()
print("✅ Listo: run_sync funcionará dentro de Colab.")

## 1. El problema: el texto libre no se deja usar

In [ ]:
import re
from pydantic_ai import Agent

agente_texto = Agent(
    "openrouter:openai/gpt-4o-mini",
    system_prompt=(
        "Eres Don Plátano, corredor estrella del Banano Exchange. "
        "En este mercado TODO se cotiza en bananas. "
        "Hablas con el dramatismo de un narrador de bolsa de valores. "
        "Responde siempre en español y en máximo 4 oraciones."
    ),
)

respuesta_texto = agente_texto.run_sync("Cotiza en bananas: la Torre Eiffel")
print(respuesta_texto.output)
print()

numeros = re.findall(r"\d[\d.,]*", respuesta_texto.output)
print("Números que encontré con la expresión regular:", numeros)

## 2. Pydantic en cinco minutos (sin IA todavía)

In [ ]:
from pydantic import BaseModel, ValidationError


class CotizacionSimple(BaseModel):
    objeto: str
    valor_en_bananas: int


cotizacion_a = CotizacionSimple(objeto="Torre Eiffel", valor_en_bananas=85000000)

print(cotizacion_a)
print("Objeto:", cotizacion_a.objeto)
print("Valor:", cotizacion_a.valor_en_bananas)
print("Como diccionario:", cotizacion_a.model_dump())

In [ ]:
# Lo razonable se convierte solito: "500" (texto) -> 500 (entero)
cotizacion_b = CotizacionSimple(objeto="Taco de pastor", valor_en_bananas="500")
print("Valor:", cotizacion_b.valor_en_bananas, "| tipo:", type(cotizacion_b.valor_en_bananas).__name__)

# Lo que no tiene sentido se rechaza con un error claro
try:
    CotizacionSimple(objeto="Auto compacto", valor_en_bananas="muchísimas")
except ValidationError as error:
    print()
    print("❌ Pydantic rechazó los datos:")
    print(error)

In [ ]:
from enum import Enum
from typing import Literal
from pydantic import Field


class Categoria(str, Enum):
    monumento = "monumento"
    comida = "comida"
    vehiculo = "vehiculo"
    objeto_cotidiano = "objeto_cotidiano"
    otro = "otro"


class Cotizacion(BaseModel):
    objeto: str = Field(
        description="Nombre del objeto o cosa que se cotiza",
    )
    categoria: Categoria = Field(
        description="Categoría que mejor describe al objeto",
    )
    valor_en_bananas: int = Field(
        gt=0,
        description="Valor estimado en bananas: un número entero positivo, sin separadores de miles",
    )
    confianza: float = Field(
        ge=0,
        le=1,
        description="Nivel de confianza entre 0 y 1, donde 0.85 significa 85 por ciento",
    )
    tendencia: Literal["alza", "baja", "estable"] = Field(
        description="Hacia dónde va el precio de este objeto en el mercado bananero",
    )
    comentario: str = Field(
        max_length=200,
        description="Comentario dramático de Don Plátano, de máximo 200 caracteres",
    )


print("✅ Formulario Cotizacion definido.")

In [ ]:
datos_base = {
    "objeto": "Torre Eiffel",
    "categoria": "monumento",
    "valor_en_bananas": 85000000,
    "confianza": 0.6,
    "tendencia": "alza",
    "comentario": "¡Sube como espuma de cerveza en final de mundial!",
}

cotizacion_valida = Cotizacion(**datos_base)

print(cotizacion_valida)
print()
print("Categoría:", cotizacion_valida.categoria.value)

In [ ]:
casos_invalidos = [
    ("valor negativo", {"valor_en_bananas": -500}),
    ("confianza como porcentaje", {"confianza": 85}),
    ("tendencia inventada", {"tendencia": "disparada"}),
    ("categoría inexistente", {"categoria": "planeta"}),
]

for nombre, cambio in casos_invalidos:
    datos = {**datos_base, **cambio}
    try:
        Cotizacion(**datos)
    except ValidationError as error:
        problema = error.errors()[0]
        print(f"❌ {nombre}: campo '{problema['loc'][0]}' -> {problema['msg']}")

## 4. Lo que realmente ve el modelo: el esquema

In [ ]:
import json

esquema = Cotizacion.model_json_schema()
print(json.dumps(esquema, indent=2, ensure_ascii=False))

## 5. Conectar el formulario con el agente: `output_type`

In [ ]:
agente_cotizador = Agent(
    "openrouter:openai/gpt-4o-mini",
    output_type=Cotizacion,
    system_prompt=(
        "Eres Don Plátano, corredor estrella del Banano Exchange. "
        "En este mercado TODO se cotiza en bananas. "
        "Estima el valor de lo que te pidan con tu mejor criterio. "
        "Tu comentario debe tener el dramatismo de un narrador de bolsa de valores "
        "y estar escrito en español."
    ),
)

resultado = agente_cotizador.run_sync("Cotiza en bananas: la Torre Eiffel")
cotizacion = resultado.output

print("Tipo de la respuesta:", type(cotizacion).__name__)
print()
print(cotizacion)

In [ ]:
print("Objeto:", cotizacion.objeto)
print("Categoría:", cotizacion.categoria.value)
print(f"Valor: {cotizacion.valor_en_bananas:,} bananas")
print(f"Confianza: {cotizacion.confianza:.0%}")
print("Tendencia:", cotizacion.tendencia)
print("Comentario:", cotizacion.comentario)
print()
print("Uso de tokens:", resultado.usage)

## 6. ¿Y si el modelo se equivoca? Reintentos

Los modelos fallan, por ejemplo devolviendo `confianza: 85` en lugar de `0.85`. Aquí PydanticAI hace algo útil: **no te entrega el error, se lo devuelve al modelo**.

```
Tú ──pregunta──▶ Modelo ──respuesta──▶ Validación (Pydantic)
                    ▲                        │
                    │      ¿válida?          │
                    │    ┌─────sí──▶ ✅ te entrega el objeto
                    │    │
                    └────┴─no──▶ le explica el error y le pide corregir
                                 (hasta agotar los reintentos)
```

Así el modelo corrige solo en la mayoría de los casos. Revisemos si la cotización anterior necesitó algún reintento, buscando en el historial los mensajes de tipo `RetryPromptPart`:

In [ ]:
from pydantic_ai.messages import RetryPromptPart

reintentos = 0
for mensaje in resultado.all_messages():
    for parte in mensaje.parts:
        if isinstance(parte, RetryPromptPart):
            reintentos += 1

print("Reintentos necesarios en esta ejecución:", reintentos)

In [ ]:
class CotizacionImposible(BaseModel):
    objeto: str
    valor_en_bananas: int = Field(gt=0, lt=1, description="Valor en bananas")


agente_imposible = Agent(
    "openrouter:openai/gpt-4o-mini",
    output_type=CotizacionImposible,
    retries=2,
)

try:
    agente_imposible.run_sync("Cotiza en bananas: la Torre Eiffel")
except Exception as error:
    print("❌ Error capturado:", type(error).__name__)
    print("Detalle (primeros 300 caracteres):")
    print(str(error)[:300])

## 7. De cotizaciones a tabla

In [ ]:
import pandas as pd

objetos = [
    "Torre Eiffel",
    "Un taco de pastor con todo",
    "Un auto compacto nuevo",
]

filas = []
for objeto in objetos:
    try:
        respuesta = agente_cotizador.run_sync(f"Cotiza en bananas: {objeto}")
        filas.append(respuesta.output.model_dump(mode="json"))
    except Exception as error:
        print(f"⚠️ No se pudo cotizar '{objeto}':", type(error).__name__)

tabla = pd.DataFrame(filas)
tabla

In [ ]:
from pathlib import Path

ruta_csv = Path("cotizaciones.csv")
tabla.to_csv(ruta_csv, index=False)

print("✅ Guardado en:", ruta_csv.resolve())

## 8. Estructura no es lo mismo que verdad

In [ ]:
valores = []
for intento in range(4):
    respuesta = agente_cotizador.run_sync("Cotiza en bananas: la Torre Eiffel")
    valores.append(respuesta.output.valor_en_bananas)

serie = pd.Series(valores)

print("Valores:", [f"{valor:,}" for valor in valores])
print(f"Mínimo: {serie.min():,} | Máximo: {serie.max():,}")
print(f"El máximo es {serie.max() / serie.min():.1f} veces el mínimo")

## 9. Errores comunes y cómo resolverlos

| Síntoma | Causa probable | Solución |
|---|---|---|
| `ValidationError` al crear tu modelo a mano | Algún dato no cumple las reglas del campo | Lee `error.errors()`: indica el campo y el motivo |
| El agente falla tras varios intentos | La salida del modelo no cumple el formulario, o el formulario es demasiado estricto o contradictorio | Revisa las `description`, relaja los límites o sube `retries` |
| `confianza` llega como 85 en lugar de 0.85 | La descripción del campo era vaga | Aclárala: "entre 0 y 1, donde 0.85 significa 85 por ciento" |
| El modelo ignora tu formato | El modelo no soporta salidas estructuradas | Prueba con otro modelo (`gpt-4o-mini` o `gemini-2.5-flash-lite`) |
| La columna de categoría muestra `Categoria.monumento` en la tabla | Usaste `model_dump()` sin `mode="json"` | Usa `model_dump(mode="json")` |
| La respuesta tarda más o cuesta más que en el Artículo 1 | El esquema se envía en cada llamada | Es normal; mantén los campos y descripciones concisos |
| Error de atributo al leer la respuesta | Versión antigua de PydanticAI (usaba `.data`) | Revisa tu versión: hoy se usa `.output` |